# RTE Challenge: small DeBERTa-v3 models + LoRA + distillation + vocabulary pruning

**Goal:** high RTE accuracy with as few parameters as possible.

**One "Run All" runs these experiments** (edit `EXPERIMENTS` in cell 2):
| Experiment | What it is |
|---|---|
| `base` | DeBERTa-v3-base + LoRA (184M params, ~100M after pruning). Reference model and **teacher**. |
| `xsmall` | DeBERTa-v3-xsmall + LoRA (71M params, ~30M after pruning). |
| `xsmall_distill` | Same xsmall model, but it also learns from the base model's predicted probabilities (knowledge distillation). |

Then every model is **shrunk** (vocabulary pruning) and cell 8 prints an **accuracy-vs-size table**.

**Before running** (right-hand panel):
1. **Settings → Accelerator → GPU T4 x2**.
2. **Settings → Internet → On** (downloads the models, `peft` and the official RTE dev set).
3. **Add Input** → your `train.jsonl` (`{"premise": ..., "hypothesis": ..., "label": "entailment", "idx": 0}`).
   A dev file is optional: without one, cell 4 downloads the official RTE dev set (277 pairs), used for evaluation only.

**Tips**
- Use **Save Version → Save & Run All (Commit)**: it runs in the background (up to 12 h) and keeps all outputs.
- Finished runs are skipped when you run again, so you can add seeds or experiments without retraining.
- Final step: set `FINAL_EXPERIMENT` to the one you pick and `SEEDS = [42, 43, 44, 45, 46]`, then Run All again.

| Cell | What it does |
|---|---|
| 1 Setup | installs `peft`, checks the GPU |
| 2 Config | experiments, seeds, hyper-parameters, data files |
| 3 Code | all functions (run once) |
| 4 Dev set | your dev file, or the official RTE dev set |
| 5 **TRAIN** | trains every experiment × seed |
| 6 **VALIDATE** | dev accuracy (mean ± std), confusion matrix, error slices |
| 7 **SHRINK** | vocabulary pruning + dev accuracy after pruning |
| 8 **COMPARE** | accuracy vs size for all experiments |
| 9 TEST | predictions for the hidden test set (only if you have a test file) |
| 10 Download | zips everything |

In [1]:
# ===== 1. SETUP =====
import os
os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"   # deterministic GPU matmuls (must be set before CUDA starts)
os.environ["TOKENIZERS_PARALLELISM"] = "false"       # avoids tokenizer fork warnings / deadlocks

!pip uninstall -y -q torchao             # Kaggle's old torchao (0.10) breaks new peft; we don't use it
!pip install -q peft sentencepiece

import torch, transformers, peft
print("torch", torch.__version__, "| transformers", transformers.__version__, "| peft", peft.__version__)
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NONE - turn on the GPU accelerator!")

torch 2.11.0+cu130 | transformers 5.20.2 | peft 0.21.2
GPU: Tesla T4


In [2]:
# ===== 2. CONFIG =====
import glob

# Default settings for every experiment (an experiment can override any of them).
CONFIG = dict(
    max_len=256,          # max tokens per (premise, hypothesis) pair
    epochs=20,
    batch_size=16,
    lr=5e-4,              # LoRA learning rate
    weight_decay=0.01,
    warmup_ratio=0.1,
    max_grad_norm=1.0,
    lora_r=8, lora_alpha=16, lora_dropout=0.1,
    val_ratio=0.1,        # 10% of TRAIN held out to pick the best epoch (the dev set is never used for this)
    split_seed=1234,      # same hold-out split for every experiment and seed
    use_amp=True,         # mixed precision on GPU; set False if you ever see NaN loss
    # knowledge distillation (only used by experiments that name a teacher)
    kd_T=2.0,             # temperature that softens the teacher's probabilities
    kd_alpha=0.7,         # weight of the teacher loss vs. the gold-label loss
    n_aug=2,              # extra perturbed copies of each training pair, labelled only by the teacher
    aug_p=0.15,           # share of words deleted or masked in a perturbed copy
)

EXPERIMENTS = [
    dict(name="base", model_name="microsoft/deberta-v3-base", seeds=[42]),     # teacher + reference
    dict(name="xsmall", model_name="microsoft/deberta-v3-xsmall"),
    dict(name="xsmall_distill", model_name="microsoft/deberta-v3-xsmall", teacher="base", epochs=10),
    # dict(name="small", model_name="microsoft/deberta-v3-small"),             # optional middle point
]
SEEDS = [42]                         # final run: [42, 43, 44, 45, 46]
FINAL_EXPERIMENT = "xsmall_distill"  # the model you submit (used by the TEST cell)
VOCAB_TOP_K = 10000                  # pruning keeps train tokens + this many most common tokens

OUT_DIR = "/kaggle/working/runs"
SUM_DIR = "/kaggle/working/summary"
PRED_DIR = "/kaggle/working/preds"

# ---- data files: found automatically under /kaggle/input; set by hand if wrong ----
INPUT_ROOT = "/kaggle/input"
def find_file(keys):
    hits = [p for p in glob.glob(os.path.join(INPUT_ROOT, "**", "*"), recursive=True)
            if os.path.isfile(p) and p.lower().endswith((".jsonl", ".json", ".tsv", ".csv"))
            and any(k in os.path.basename(p).lower() for k in keys)]
    return sorted(hits)[0] if hits else None

TRAIN_FILE = find_file(["train"])
DEV_FILE   = find_file(["val", "dev"])
TEST_FILE  = find_file(["test"])
# TRAIN_FILE = "/kaggle/input/your-dataset/train.jsonl"   # <- example of setting a path by hand
print("TRAIN_FILE:", TRAIN_FILE); print("DEV_FILE:  ", DEV_FILE); print("TEST_FILE: ", TEST_FILE)
assert TRAIN_FILE, "Could not find a train file - set TRAIN_FILE by hand above."

TRAIN_FILE: /kaggle/input/datasets/prakashroy1211/dl-challenge-train/train.jsonl
DEV_FILE:   None
TEST_FILE:  None


In [3]:
# ===== 3. CODE: all functions (run once) =====
import csv, json, random, re, statistics, time, importlib
import numpy as np
import torch
import torch.nn.functional as F
from torch import nn
from torch.utils.data import DataLoader, Dataset
from transformers import AutoModelForSequenceClassification, AutoTokenizer, get_linear_schedule_with_warmup
from peft import LoraConfig, PeftModel, TaskType, get_peft_model

# Kaggle ships torchao 0.10, which new peft versions reject even though we never use it.
def _no_torchao(): return False
for _m in ("peft.import_utils", "peft.tuners.lora.torchao", "peft.utils.quantization_utils"):
    try: setattr(importlib.import_module(_m), "is_torchao_available", _no_torchao)
    except Exception: pass

LABEL2ID = {"entailment": 0, "not_entailment": 1}
ID2LABEL = {0: "entailment", 1: "not_entailment"}
MODEL_KEYS = ("input_ids", "attention_mask", "token_type_ids")
# LoRA on Q, K, V, attention-output, FFN-in, FFN-out of every transformer layer
LORA_TARGETS = (r".*encoder\.layer\.\d+\.(attention\.self\.(query_proj|key_proj|value_proj)"
                r"|attention\.output\.dense|intermediate\.dense|output\.dense)")
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
for d in (OUT_DIR, SUM_DIR, PRED_DIR): os.makedirs(d, exist_ok=True)


def set_seed(seed):
    random.seed(seed); np.random.seed(seed)
    torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    torch.use_deterministic_algorithms(True, warn_only=True)


def exp_cfg(exp): return {**CONFIG, **exp}
def get_exp(name): return next(e for e in EXPERIMENTS if e["name"] == name)
def run_path(exp, seed): return os.path.join(OUT_DIR, f"{exp['name']}_s{seed}")
def amp_dtype():
    if DEVICE.type != "cuda" or not CONFIG.get("use_amp", True): return None
    return torch.bfloat16 if torch.cuda.get_device_capability()[0] >= 8 else torch.float16


# ---------------- data ----------------
def parse_label(raw):
    if raw is None or raw == "" or str(raw) == "-1": return None          # unlabelled
    if isinstance(raw, str) and raw.strip() in LABEL2ID: return LABEL2ID[raw.strip()]
    return int(raw)


def read_examples(path):
    ext = os.path.splitext(path)[1].lower()
    if ext in (".tsv", ".csv"):
        with open(path, encoding="utf-8", newline="") as f:
            recs = list(csv.DictReader(f, delimiter="\t" if ext == ".tsv" else ",",
                                       quoting=csv.QUOTE_NONE if ext == ".tsv" else csv.QUOTE_MINIMAL))
    else:
        text = open(path, encoding="utf-8").read().strip()
        try:
            recs = json.loads(text); recs = recs if isinstance(recs, list) else recs.get("data", [])
        except json.JSONDecodeError:
            recs = [json.loads(l) for l in text.splitlines() if l.strip()]
    return [{"idx": r.get("idx", r.get("index", i)),
             "premise": str(r.get("premise", r.get("sentence1"))).strip(),
             "hypothesis": str(r.get("hypothesis", r.get("sentence2"))).strip(),
             "label": parse_label(r.get("label"))} for i, r in enumerate(recs)]


def write_jsonl(path, examples):
    os.makedirs(os.path.dirname(path), exist_ok=True)
    with open(path, "w", encoding="utf-8") as f:
        for e in examples:
            lab = ID2LABEL[e["label"]] if isinstance(e["label"], int) else e["label"]
            f.write(json.dumps({"premise": e["premise"], "hypothesis": e["hypothesis"], "label": lab, "idx": e["idx"]}) + "\n")


class RTEDataset(Dataset):
    # Model input = [CLS] premise [SEP] hypothesis [SEP], truncated to max_len (premise cut first).
    def __init__(self, examples, tokenizer, max_len):
        enc = tokenizer([e["premise"] for e in examples], [e["hypothesis"] for e in examples],
                        truncation=True, max_length=max_len)
        self.n_truncated = sum(len(x) >= max_len for x in enc["input_ids"])
        keys = [k for k in MODEL_KEYS if k in enc]
        self.items = []
        for i, e in enumerate(examples):
            item = {k: enc[k][i] for k in keys}
            if e["label"] is not None: item["labels"] = e["label"]
            if "teacher" in e: item["teacher_logits"] = e["teacher"]
            self.items.append(item)
    def __len__(self): return len(self.items)
    def __getitem__(self, i): return self.items[i]


def make_collate(tokenizer):
    def collate(batch):                    # dynamic padding; unlabelled items get label -100
        out = dict(tokenizer.pad([{k: b[k] for k in MODEL_KEYS if k in b} for b in batch],
                                 padding=True, return_tensors="pt"))
        out["labels"] = torch.tensor([b.get("labels", -100) for b in batch], dtype=torch.long)
        if "teacher_logits" in batch[0]:
            out["teacher_logits"] = torch.tensor([b["teacher_logits"] for b in batch], dtype=torch.float)
        return out
    return collate


def stratified_split(examples, ratio, seed):
    rng = random.Random(seed); train, hold = [], []
    for lab in (0, 1):
        group = [e for e in examples if e["label"] == lab]; rng.shuffle(group)
        n = int(round(len(group) * ratio)); hold += group[:n]; train += group[n:]
    rng.shuffle(train)
    return train, hold


def augment(examples, n, p, mask_token, seed):
    # Perturbed copies for distillation: each word is deleted (p/2) or masked (p/2). No gold label:
    # the teacher's probabilities are the only target, since a perturbation can change the answer.
    rng = random.Random(seed + 7)
    def perturb(text):
        out = []
        for w in text.split():
            r = rng.random()
            if r < p / 2: continue
            out.append(mask_token if r < p else w)
        return " ".join(out) if out else text
    return [{"idx": f"aug{k}_{e['idx']}", "premise": perturb(e["premise"]),
             "hypothesis": perturb(e["hypothesis"]), "label": None} for k in range(n) for e in examples]


# ---------------- model ----------------
def build_lora_model(cfg):
    model = AutoModelForSequenceClassification.from_pretrained(
        cfg["model_name"], num_labels=2, id2label=ID2LABEL, label2id=LABEL2ID).float()  # fp32 master weights
    base_params = sum(p.numel() for p in model.parameters())
    lcfg = LoraConfig(task_type=TaskType.SEQ_CLS, r=cfg["lora_r"], lora_alpha=cfg["lora_alpha"],
                      lora_dropout=cfg["lora_dropout"], target_modules=LORA_TARGETS,
                      modules_to_save=["classifier", "pooler"])   # new head layers are fully trained
    return get_peft_model(model, lcfg), base_params


def load_trained(run_dir):
    best_dir = os.path.join(run_dir, "best")
    rc = json.load(open(os.path.join(best_dir, "run_config.json")))
    base = AutoModelForSequenceClassification.from_pretrained(
        rc["model_name"], num_labels=2, id2label=ID2LABEL, label2id=LABEL2ID).float()
    model = PeftModel.from_pretrained(base, best_dir).merge_and_unload()   # LoRA folded into the weights
    return model.to(DEVICE).eval(), AutoTokenizer.from_pretrained(best_dir), rc


@torch.no_grad()
def evaluate(model, loader):
    model.eval(); tot, correct, n = 0.0, 0, 0
    for batch in loader:
        labels = batch["labels"].to(DEVICE)
        inputs = {k: batch[k].to(DEVICE) for k in MODEL_KEYS if k in batch}
        with torch.autocast(DEVICE.type, dtype=amp_dtype(), enabled=amp_dtype() is not None):
            logits = model(**inputs).logits.float()
        tot += F.cross_entropy(logits, labels, reduction="sum").item()
        correct += (logits.argmax(-1) == labels).sum().item(); n += labels.numel()
    return tot / n, correct / n


@torch.no_grad()
def predict_logits(model, loader, remap=None):
    model.eval(); out = []
    for batch in loader:
        inputs = {k: batch[k].to(DEVICE) for k in MODEL_KEYS if k in batch}
        if remap is not None: inputs["input_ids"] = remap[inputs["input_ids"]]   # pruned vocabulary
        with torch.autocast(DEVICE.type, dtype=amp_dtype(), enabled=amp_dtype() is not None):
            out += model(**inputs).logits.float().cpu().tolist()
    return out


def p_entail(logits): return torch.softmax(torch.tensor(logits), -1)[:, 0].tolist()


def chosen_run(exp):
    # Best run of an experiment by HOLD-OUT accuracy (decided without looking at dev).
    res = [json.load(open(p)) for p in glob.glob(os.path.join(OUT_DIR, "*", "results.json"))]
    res = [r for r in res if r["experiment"] == exp["name"]]
    assert res, f"No trained runs for experiment '{exp['name']}' - run the TRAIN cell first."
    return os.path.join(OUT_DIR, max(res, key=lambda r: (r["holdout_acc"], -r["holdout_loss"]))["run"])


# ---------------- training ----------------
def train_one(exp, seed):
    cfg = exp_cfg(exp); out_dir = run_path(exp, seed)
    if os.path.exists(os.path.join(out_dir, "results.json")):
        print(f"skip {os.path.basename(out_dir)}: already trained"); return json.load(open(os.path.join(out_dir, "results.json")))
    set_seed(seed); os.makedirs(out_dir, exist_ok=True)
    tokenizer = AutoTokenizer.from_pretrained(cfg["model_name"])
    examples = [e for e in read_examples(TRAIN_FILE) if e["label"] is not None]
    train_ex, hold_ex = stratified_split(examples, cfg["val_ratio"], cfg["split_seed"])

    if cfg.get("teacher"):                 # knowledge distillation: add the teacher's logits as soft targets
        t_run = chosen_run(get_exp(cfg["teacher"]))
        train_ex = train_ex + augment(train_ex, cfg["n_aug"], cfg["aug_p"], tokenizer.mask_token, seed)
        teacher, t_tok, t_rc = load_trained(t_run)
        t_dl = DataLoader(RTEDataset(train_ex, t_tok, t_rc["max_len"]), batch_size=64, collate_fn=make_collate(t_tok))
        for e, l in zip(train_ex, predict_logits(teacher, t_dl)): e["teacher"] = l
        del teacher; torch.cuda.empty_cache()
        print(f"teacher: {os.path.basename(t_run)} -> soft targets for {len(train_ex)} pairs (incl. perturbed copies)")

    collate = make_collate(tokenizer)
    train_dl = DataLoader(RTEDataset(train_ex, tokenizer, cfg["max_len"]), batch_size=cfg["batch_size"],
                          shuffle=True, collate_fn=collate, generator=torch.Generator().manual_seed(seed))
    hold_dl = DataLoader(RTEDataset(hold_ex, tokenizer, cfg["max_len"]), batch_size=64, collate_fn=collate)

    model, base_params = build_lora_model(cfg); model.to(DEVICE)
    trainable = [p for p in model.parameters() if p.requires_grad]
    n_train = sum(p.numel() for p in trainable)
    print(f"\n=== {exp['name']} seed {seed} | {cfg['model_name']} | train {len(train_ex)} | hold-out {len(hold_ex)} | "
          f"params {base_params:,} total, {n_train:,} trainable ===")

    decay = [p for p in trainable if p.ndim >= 2]; no_decay = [p for p in trainable if p.ndim < 2]
    opt = torch.optim.AdamW([{"params": decay, "weight_decay": cfg["weight_decay"]},
                             {"params": no_decay, "weight_decay": 0.0}], lr=cfg["lr"], eps=1e-6)
    steps = len(train_dl) * cfg["epochs"]
    sched = get_linear_schedule_with_warmup(opt, int(cfg["warmup_ratio"] * steps), steps)
    dtype = amp_dtype(); scaler = torch.amp.GradScaler("cuda", enabled=dtype == torch.float16)
    T, alpha = cfg["kd_T"], cfg["kd_alpha"]

    log, best, best_state = [], None, None
    for epoch in range(1, cfg["epochs"] + 1):
        model.train(); t0, run_loss, seen = time.time(), 0.0, 0
        for batch in train_dl:
            labels = batch["labels"].to(DEVICE)
            inputs = {k: batch[k].to(DEVICE) for k in MODEL_KEYS if k in batch}
            with torch.autocast(DEVICE.type, dtype=dtype, enabled=dtype is not None):
                logits = model(**inputs).logits.float()
            has_label = labels != -100
            ce = F.cross_entropy(logits[has_label], labels[has_label]) if has_label.any() else logits.sum() * 0
            if "teacher_logits" in batch:   # loss = alpha * T^2 * KL(teacher || student) + (1 - alpha) * CE
                t = batch["teacher_logits"].to(DEVICE)
                kd = F.kl_div(F.log_softmax(logits / T, -1), F.softmax(t / T, -1), reduction="batchmean") * T * T
                loss = alpha * kd + (1 - alpha) * ce
            else:
                loss = ce                   # plain cross-entropy
            opt.zero_grad(set_to_none=True)
            scaler.scale(loss).backward(); scaler.unscale_(opt)
            torch.nn.utils.clip_grad_norm_(trainable, cfg["max_grad_norm"])
            scaler.step(opt); scaler.update(); sched.step()
            run_loss += loss.item() * labels.size(0); seen += labels.size(0)
        h_loss, h_acc = evaluate(model, hold_dl)
        row = {"epoch": epoch, "train_loss": run_loss / seen, "holdout_loss": h_loss, "holdout_acc": h_acc}
        log.append(row)
        better = best is None or (h_acc, -h_loss) > (best["holdout_acc"], -best["holdout_loss"])
        if better:
            best = row; best_state = {n: p.detach().cpu().clone() for n, p in model.named_parameters() if p.requires_grad}
        print(f"epoch {epoch:2d} | train loss {row['train_loss']:.4f} | hold-out loss {h_loss:.4f} "
              f"acc {h_acc:.4f} | {time.time() - t0:.0f}s" + ("  <- best" if better else ""))
        if not np.isfinite(row["train_loss"]):
            print('NaN loss - set CONFIG["use_amp"] = False and rerun'); break

    with torch.no_grad():                  # restore the best epoch, then save adapter + head + tokenizer
        for n, p in model.named_parameters():
            if n in best_state: p.copy_(best_state[n].to(p.device))
    best_dir = os.path.join(out_dir, "best")
    model.save_pretrained(best_dir); tokenizer.save_pretrained(best_dir)
    json.dump({**cfg, "seed": seed, "lora_targets": LORA_TARGETS}, open(os.path.join(best_dir, "run_config.json"), "w"), indent=2)
    json.dump(log, open(os.path.join(out_dir, "train_log.json"), "w"), indent=2)
    result = {"experiment": exp["name"], "run": os.path.basename(out_dir), "seed": seed, "model_name": cfg["model_name"],
              "best_epoch": best["epoch"], "holdout_acc": best["holdout_acc"], "holdout_loss": best["holdout_loss"],
              "total_params": base_params, "trainable_params": n_train}
    json.dump(result, open(os.path.join(out_dir, "results.json"), "w"), indent=2)   # written last = run finished
    del model, opt; torch.cuda.empty_cache()
    return result


# ---------------- vocabulary pruning ----------------
def prune_vocab(model, keep_ids, unk_id, pad_id):
    # Keep only the embedding rows in keep_ids (sorted old ids). Returns the model and an old->new id table.
    emb = model.get_input_embeddings()
    keep = torch.tensor(keep_ids, dtype=torch.long, device=emb.weight.device)
    new = nn.Embedding(len(keep_ids), emb.embedding_dim, padding_idx=keep_ids.index(pad_id))
    new.weight.data.copy_(emb.weight.data[keep])
    model.set_input_embeddings(new.to(emb.weight.device)); model.config.vocab_size = len(keep_ids)
    remap = torch.full((emb.num_embeddings,), keep_ids.index(unk_id), dtype=torch.long)
    remap[keep.cpu()] = torch.arange(len(keep_ids))
    return model, remap.to(DEVICE)


def load_pruned(pruned_dir):
    model = AutoModelForSequenceClassification.from_pretrained(pruned_dir).float().to(DEVICE).eval()
    tok = AutoTokenizer.from_pretrained(pruned_dir)
    meta = json.load(open(os.path.join(pruned_dir, "vocab_keep.json")))
    remap = torch.full((meta["original_vocab_size"],), meta["keep_ids"].index(tok.unk_token_id), dtype=torch.long)
    remap[torch.tensor(meta["keep_ids"])] = torch.arange(len(meta["keep_ids"]))
    return model, tok, meta, remap.to(DEVICE)


# ---------------- outputs + error analysis ----------------
def write_predictions(path_jsonl, examples, preds, probs):
    os.makedirs(os.path.dirname(path_jsonl), exist_ok=True)
    with open(path_jsonl, "w") as f:
        for e, p in zip(examples, preds): f.write(json.dumps({"idx": e["idx"], "label": ID2LABEL[p]}) + "\n")
    with open(path_jsonl.replace(".jsonl", "_detailed.csv"), "w", newline="", encoding="utf-8") as f:
        w = csv.writer(f); w.writerow(["idx", "premise", "hypothesis", "gold", "pred", "p_entailment"])
        for e, p, pr in zip(examples, preds, probs):
            w.writerow([e["idx"], e["premise"], e["hypothesis"], ID2LABEL.get(e["label"], ""), ID2LABEL[p], round(pr, 4)])


NEG = {"not", "no", "never", "n't", "none", "nobody", "nothing", "neither", "nor", "without", "cannot", "yet"}
def words(t): return re.findall(r"[a-z0-9]+(?:'[a-z]+)?", t.lower())

def slice_accuracy(examples, preds):
    def acc(mask):
        sel = [p == e["label"] for e, p, m in zip(examples, preds, mask) if m]
        return (len(sel), sum(sel) / len(sel) if sel else float("nan"))
    plen = [len(words(e["premise"])) for e in examples]
    ovl = [len(set(words(e["hypothesis"])) & set(words(e["premise"]))) / max(len(set(words(e["hypothesis"]))), 1) for e in examples]
    neg = [bool(NEG & set(words(e["premise"] + " " + e["hypothesis"]))) for e in examples]
    return {"has negation": acc(neg), "no negation": acc([not x for x in neg]),
            "premise <= 30 words": acc([l <= 30 for l in plen]), "premise 31-60 words": acc([30 < l <= 60 for l in plen]),
            "premise > 60 words": acc([l > 60 for l in plen]),
            "word overlap >= 0.8": acc([o >= 0.8 for o in ovl]), "word overlap < 0.8": acc([o < 0.8 for o in ovl]),
            "gold = entailment": acc([e["label"] == 0 for e in examples]),
            "gold = not_entailment": acc([e["label"] == 1 for e in examples])}

print("functions ready | device:", DEVICE)

functions ready | device: cuda


In [4]:
# ===== 4. DEV SET =====
# The course dev set (~277 pairs) is the official RTE validation split. If you have no dev file, it is downloaded
# from Hugging Face (GLUE RTE validation). It is used ONLY to measure accuracy, never for training.
def norm(t): return " ".join(t.lower().split())

train_all = read_examples(TRAIN_FILE)
print(f"train file: {len(train_all)} examples ({TRAIN_FILE})")
if DEV_FILE is not None:
    print(f"Using your dev file: {DEV_FILE} ({len(read_examples(DEV_FILE))} examples)")
else:
    dev = None
    try:
        from datasets import load_dataset
        for name in ("nyu-mll/glue", "glue"):
            try:
                ds = load_dataset(name, "rte", split="validation")
                dev = [{"idx": r["idx"], "premise": r["sentence1"], "hypothesis": r["sentence2"],
                        "label": ds.features["label"].int2str(r["label"])} for r in ds]
                print(f"Downloaded official RTE dev set from '{name}': {len(dev)} examples"); break
            except Exception as err:
                print(f"  could not load '{name}': {type(err).__name__}: {str(err)[:150]}")
    except ImportError:
        print("  the `datasets` library is not installed")
    if dev:
        train_keys = {(norm(e["premise"]), norm(e["hypothesis"])) for e in train_all}
        overlap = [d for d in dev if (norm(d["premise"]), norm(d["hypothesis"])) in train_keys]
        if overlap:
            print(f"  WARNING: {len(overlap)} dev pairs also appear in train.jsonl - removed from dev")
            dev = [d for d in dev if (norm(d["premise"]), norm(d["hypothesis"])) not in train_keys]
        else:
            print("  overlap check: no dev pair appears in train.jsonl")
        DEV_FILE = "/kaggle/working/data/dev.jsonl"; write_jsonl(DEV_FILE, dev)
    else:   # no internet: carve a stand-in dev set out of train and train on the rest
        labelled = [e for e in train_all if e["label"] is not None]
        rest, dev = stratified_split(labelled, min(277 / len(labelled), 0.2), seed=2024)
        TRAIN_FILE, DEV_FILE = "/kaggle/working/data/train_minus_dev.jsonl", "/kaggle/working/data/dev_from_train.jsonl"
        write_jsonl(TRAIN_FILE, rest); write_jsonl(DEV_FILE, dev)
        print(f"WARNING: could not download the official dev set (is Internet on?). Using {len(dev)} train examples "
              f"as a stand-in dev set; training on the other {len(rest)}. Not comparable to the instructor's baseline.")
print("TRAIN_FILE =", TRAIN_FILE); print("DEV_FILE   =", DEV_FILE)

train file: 2490 examples (/kaggle/input/datasets/prakashroy1211/dl-challenge-train/train.jsonl)


README.md:   0%|          | 0.00/35.3k [00:00<?, ?B/s]

rte/train-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B /  584kB            

rte/train-00000-of-00001.parquet: downloading bytes:           |  0.00B            

rte/validation-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 69.0kB            

rte/validation-00000-of-00001.parquet: downloading bytes:           |  0.00B            

... (2 outputs left out)

Generating train split:   0%|          | 0/2490 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/277 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/3000 [00:00<?, ? examples/s]

Downloaded official RTE dev set from 'nyu-mll/glue': 277 examples
  overlap check: no dev pair appears in train.jsonl
TRAIN_FILE = /kaggle/input/datasets/prakashroy1211/dl-challenge-train/train.jsonl
DEV_FILE   = /kaggle/working/data/dev.jsonl


In [5]:
# ===== 5. TRAIN: every experiment x seed (finished runs are skipped) =====
# Teachers must come before their students in EXPERIMENTS.
for exp in EXPERIMENTS:
    for seed in exp.get("seeds", SEEDS):
        train_one(exp, seed)

print("\nTrained runs:")
for p in sorted(glob.glob(os.path.join(OUT_DIR, "*", "results.json"))):
    r = json.load(open(p))
    print(f"  {r['run']:<24} best epoch {r['best_epoch']:>2}  hold-out acc {r['holdout_acc']:.4f}  "
          f"params {r['total_params']:,} ({r['trainable_params']:,} trainable)")

config.json:   0%|          | 0.00/579 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

spm.model: reconstructing file:   0%|          |  0.00B / 2.46MB            

spm.model: downloading bytes:           |  0.00B            

pytorch_model.bin: reconstructing file:   0%|          |  0.00B /  371MB            

pytorch_model.bin: downloading bytes:           |  0.00B            

... (16 outputs left out)

teacher: base_s42 -> soft targets for 6723 pairs (incl. perturbed copies)


Loading weights:   0%|          | 0/198 [00:00<?, ?it/s]

[transformers] DebertaV2ForSequenceClassification LOAD REPORT from: microsoft/deberta-v3-xsmall
Key                                        | Status     | 
-------------------------------------------+------------+-
lm_predictions.lm_head.LayerNorm.weight    | UNEXPECTED | 
mask_predictions.LayerNorm.weight          | UNEXPECTED | 
lm_predictions.lm_head.bias                | UNEXPECTED | 
mask_predictions.classifier.weight         | UNEXPECTED | 
mask_predictions.LayerNorm.bias            | UNEXPECTED | 
deberta.embeddings.word_embeddings._weight | UNEXPECTED | 
lm_predictions.lm_head.dense.weight        | UNEXPECTED | 
lm_predictions.lm_head.LayerNorm.bias      | UNEXPECTED | 
lm_predictions.lm_head.dense.bias          | UNEXPECTED | 
mask_predictions.classifier.bias           | UNEXPECTED | 
mask_predictions.dense.bias                | UNEXPECTED | 
mask_predictions.dense.weight              | UNEXPECTED | 
classifier.weight                          | MISSING    | 
pooler.dense.bias  


=== xsmall_distill seed 42 | microsoft/deberta-v3-xsmall | train 6723 | hold-out 249 | params 70,830,722 total, 812,162 trainable ===
epoch  1 | train loss 1.8130 | hold-out loss 0.7835 acc 0.6667 | 156s  <- best
epoch  2 | train loss 1.2574 | hold-out loss 0.7087 acc 0.7189 | 156s  <- best
epoch  3 | train loss 0.8369 | hold-out loss 1.2047 acc 0.7510 | 157s  <- best
epoch  4 | train loss 0.5903 | hold-out loss 1.3555 acc 0.7550 | 155s  <- best
epoch  5 | train loss 0.4464 | hold-out loss 1.2636 acc 0.7952 | 155s  <- best
epoch  6 | train loss 0.3040 | hold-out loss 1.4738 acc 0.8032 | 155s  <- best
epoch  7 | train loss 0.2565 | hold-out loss 1.8930 acc 0.7952 | 154s
epoch  8 | train loss 0.2032 | hold-out loss 2.0134 acc 0.7831 | 154s
epoch  9 | train loss 0.1592 | hold-out loss 1.7763 acc 0.8072 | 153s  <- best
epoch 10 | train loss 0.1442 | hold-out loss 2.1200 acc 0.7751 | 155s

Trained runs:
  base_s42                 best epoch 20  hold-out acc 0.8956  params 184,423,682 (1,91

In [6]:
# ===== 6. VALIDATE: dev accuracy for every experiment =====
dev_examples = read_examples(DEV_FILE)
assert all(e["label"] is not None for e in dev_examples), "DEV_FILE has no labels"
golds = [e["label"] for e in dev_examples]
all_results = [json.load(open(p)) for p in glob.glob(os.path.join(OUT_DIR, "*", "results.json"))]
dev_outputs = {}

for exp in EXPERIMENTS:
    runs = sorted((r for r in all_results if r["experiment"] == exp["name"]), key=lambda r: r["seed"])
    if not runs: print(f"{exp['name']}: not trained yet"); continue
    accs = {}
    for r in runs:
        model, tok, rc = load_trained(os.path.join(OUT_DIR, r["run"]))
        dl = DataLoader(RTEDataset(dev_examples, tok, rc["max_len"]), batch_size=64, collate_fn=make_collate(tok))
        probs = p_entail(predict_logits(model, dl)); del model; torch.cuda.empty_cache()
        preds = [0 if p >= 0.5 else 1 for p in probs]
        accs[r["run"]] = sum(p == g for p, g in zip(preds, golds)) / len(golds)
        dev_outputs[r["run"]] = (preds, probs)
        write_predictions(os.path.join(PRED_DIR, r["run"], "dev_predictions.jsonl"), dev_examples, preds, probs)
    vals = list(accs.values()); best = os.path.basename(chosen_run(exp))
    mean, sd = statistics.mean(vals), (statistics.stdev(vals) if len(vals) > 1 else 0.0)
    print(f"\n### {exp['name']}  ({exp_cfg(exp)['model_name']})")
    for k, v in accs.items(): print(f"  {k:<24} dev accuracy {v:.4f}" + ("   <- chosen (best hold-out)" if k == best else ""))
    print(f"  DEV ACCURACY: {100 * mean:.2f} +- {100 * sd:.2f} %  over {len(vals)} seed(s)")
    preds, probs = dev_outputs[best]
    cm = [[sum(1 for p, g in zip(preds, golds) if g == a and p == b) for b in (0, 1)] for a in (0, 1)]
    print(f"  confusion (rows gold, cols pred): entailment {cm[0]}  not_entailment {cm[1]}")
    print("  " + " | ".join(f"{k}: {a:.3f} (n={n})" for k, (n, a) in slice_accuracy(dev_examples, preds).items()))
    json.dump({"experiment": exp["name"], "model_name": exp_cfg(exp)["model_name"], "dev_acc_per_run": accs,
               "mean": mean, "std": sd, "chosen_run": best,
               "total_params": runs[0]["total_params"], "trainable_params": runs[0]["trainable_params"],
               "holdout_acc_mean": statistics.mean(r["holdout_acc"] for r in runs)},
              open(os.path.join(SUM_DIR, f"dev_{exp['name']}.json"), "w"), indent=2)

if FINAL_EXPERIMENT in [e["name"] for e in EXPERIMENTS] and os.path.exists(os.path.join(SUM_DIR, f"dev_{FINAL_EXPERIMENT}.json")):
    best = os.path.basename(chosen_run(get_exp(FINAL_EXPERIMENT))); preds, probs = dev_outputs[best]
    print(f"\nSome mistakes of {best}:")
    for e, p, pr in [(e, p, pr) for e, p, pr in zip(dev_examples, preds, probs) if p != e["label"]][:8]:
        print(f"- [gold {ID2LABEL[e['label']]}, pred {ID2LABEL[p]}, P(ent)={pr:.2f}]\n  P: {e['premise'][:200]}\n  H: {e['hypothesis']}")

Loading weights:   0%|          | 0/198 [00:00<?, ?it/s]

[transformers] DebertaV2ForSequenceClassification LOAD REPORT from: microsoft/deberta-v3-base
Key                                     | Status     | 
----------------------------------------+------------+-
lm_predictions.lm_head.LayerNorm.weight | UNEXPECTED | 
mask_predictions.LayerNorm.weight       | UNEXPECTED | 
lm_predictions.lm_head.bias             | UNEXPECTED | 
mask_predictions.classifier.weight      | UNEXPECTED | 
mask_predictions.LayerNorm.bias         | UNEXPECTED | 
lm_predictions.lm_head.dense.weight     | UNEXPECTED | 
lm_predictions.lm_head.LayerNorm.bias   | UNEXPECTED | 
lm_predictions.lm_head.dense.bias       | UNEXPECTED | 
mask_predictions.classifier.bias        | UNEXPECTED | 
mask_predictions.dense.bias             | UNEXPECTED | 
mask_predictions.dense.weight           | UNEXPECTED | 
classifier.weight                       | MISSING    | 
pooler.dense.bias                       | MISSING    | 
classifier.bias                         | MISSING    | 
pooler.den


### base  (microsoft/deberta-v3-base)
  base_s42                 dev accuracy 0.8628   <- chosen (best hold-out)
  DEV ACCURACY: 86.28 +- 0.00 %  over 1 seed(s)
  confusion (rows gold, cols pred): entailment [130, 16]  not_entailment [22, 109]
  has negation: 0.775 (n=40) | no negation: 0.878 (n=237) | premise <= 30 words: 0.852 (n=135) | premise 31-60 words: 0.900 (n=80) | premise > 60 words: 0.839 (n=62) | word overlap >= 0.8: 0.862 (n=87) | word overlap < 0.8: 0.863 (n=190) | gold = entailment: 0.890 (n=146) | gold = not_entailment: 0.832 (n=131)


Loading weights:   0%|          | 0/198 [00:00<?, ?it/s]

[transformers] DebertaV2ForSequenceClassification LOAD REPORT from: microsoft/deberta-v3-xsmall
Key                                        | Status     | 
-------------------------------------------+------------+-
lm_predictions.lm_head.LayerNorm.weight    | UNEXPECTED | 
mask_predictions.LayerNorm.weight          | UNEXPECTED | 
lm_predictions.lm_head.bias                | UNEXPECTED | 
mask_predictions.classifier.weight         | UNEXPECTED | 
mask_predictions.LayerNorm.bias            | UNEXPECTED | 
deberta.embeddings.word_embeddings._weight | UNEXPECTED | 
lm_predictions.lm_head.dense.weight        | UNEXPECTED | 
lm_predictions.lm_head.LayerNorm.bias      | UNEXPECTED | 
lm_predictions.lm_head.dense.bias          | UNEXPECTED | 
mask_predictions.classifier.bias           | UNEXPECTED | 
mask_predictions.dense.bias                | UNEXPECTED | 
mask_predictions.dense.weight              | UNEXPECTED | 
classifier.weight                          | MISSING    | 
pooler.dense.bias  


### xsmall  (microsoft/deberta-v3-xsmall)
  xsmall_s42               dev accuracy 0.7726   <- chosen (best hold-out)
  DEV ACCURACY: 77.26 +- 0.00 %  over 1 seed(s)
  confusion (rows gold, cols pred): entailment [119, 27]  not_entailment [36, 95]
  has negation: 0.675 (n=40) | no negation: 0.789 (n=237) | premise <= 30 words: 0.800 (n=135) | premise 31-60 words: 0.738 (n=80) | premise > 60 words: 0.758 (n=62) | word overlap >= 0.8: 0.782 (n=87) | word overlap < 0.8: 0.768 (n=190) | gold = entailment: 0.815 (n=146) | gold = not_entailment: 0.725 (n=131)


Loading weights:   0%|          | 0/198 [00:00<?, ?it/s]

[transformers] DebertaV2ForSequenceClassification LOAD REPORT from: microsoft/deberta-v3-xsmall
Key                                        | Status     | 
-------------------------------------------+------------+-
lm_predictions.lm_head.LayerNorm.weight    | UNEXPECTED | 
mask_predictions.LayerNorm.weight          | UNEXPECTED | 
lm_predictions.lm_head.bias                | UNEXPECTED | 
mask_predictions.classifier.weight         | UNEXPECTED | 
mask_predictions.LayerNorm.bias            | UNEXPECTED | 
deberta.embeddings.word_embeddings._weight | UNEXPECTED | 
lm_predictions.lm_head.dense.weight        | UNEXPECTED | 
lm_predictions.lm_head.LayerNorm.bias      | UNEXPECTED | 
lm_predictions.lm_head.dense.bias          | UNEXPECTED | 
mask_predictions.classifier.bias           | UNEXPECTED | 
mask_predictions.dense.bias                | UNEXPECTED | 
mask_predictions.dense.weight              | UNEXPECTED | 
classifier.weight                          | MISSING    | 
pooler.dense.bias  


### xsmall_distill  (microsoft/deberta-v3-xsmall)
  xsmall_distill_s42       dev accuracy 0.7798   <- chosen (best hold-out)
  DEV ACCURACY: 77.98 +- 0.00 %  over 1 seed(s)
  confusion (rows gold, cols pred): entailment [128, 18]  not_entailment [43, 88]
  has negation: 0.650 (n=40) | no negation: 0.802 (n=237) | premise <= 30 words: 0.807 (n=135) | premise 31-60 words: 0.750 (n=80) | premise > 60 words: 0.758 (n=62) | word overlap >= 0.8: 0.793 (n=87) | word overlap < 0.8: 0.774 (n=190) | gold = entailment: 0.877 (n=146) | gold = not_entailment: 0.672 (n=131)

Some mistakes of xsmall_distill_s42:
- [gold not_entailment, pred entailment, P(ent)=1.00]
  P: Dana Reeve, the widow of the actor Christopher Reeve, has died of lung cancer at age 44, according to the Christopher Reeve Foundation.
  H: Christopher Reeve had an accident.
- [gold entailment, pred not_entailment, P(ent)=0.00]
  P: Yet, we now are discovering that antibiotics are losing their effectiveness against illness. Disease

In [7]:
# ===== 7. SHRINK: vocabulary pruning for every experiment =====
# Most parameters of these models are the 128,100-token embedding table, but RTE text uses only a small part of it.
# We keep the rows for every token in the TRAINING data plus the VOCAB_TOP_K most common tokens; any other token
# becomes [UNK]. The dev set is not used to choose the vocabulary, so the dev score is an honest estimate for test data.
dev_examples = read_examples(DEV_FILE); golds = [e["label"] for e in dev_examples]
for exp in EXPERIMENTS:
    if not os.path.exists(os.path.join(SUM_DIR, f"dev_{exp['name']}.json")): continue
    run = chosen_run(exp)
    model, tok, rc = load_trained(run)
    full_params = sum(p.numel() for p in model.parameters()); V = model.get_input_embeddings().num_embeddings
    train_ids = set()
    for e in read_examples(TRAIN_FILE): train_ids.update(tok(e["premise"], e["hypothesis"])["input_ids"])
    keep = sorted(i for i in (train_ids | set(range(min(VOCAB_TOP_K, V))) | set(tok.all_special_ids)) if i < V)
    model, remap = prune_vocab(model, keep, tok.unk_token_id, tok.pad_token_id)
    pruned_params = sum(p.numel() for p in model.parameters())
    dl = DataLoader(RTEDataset(dev_examples, tok, rc["max_len"]), batch_size=64, collate_fn=make_collate(tok))
    probs = p_entail(predict_logits(model, dl, remap)); preds = [0 if p >= 0.5 else 1 for p in probs]
    pruned_acc = sum(p == g for p, g in zip(preds, golds)) / len(golds)
    full_acc = json.load(open(os.path.join(SUM_DIR, f"dev_{exp['name']}.json")))["dev_acc_per_run"][os.path.basename(run)]
    keep_set = set(keep); dev_tok = [i for e in dev_examples for i in tok(e["premise"], e["hypothesis"])["input_ids"]]
    oov = sum(i not in keep_set for i in dev_tok) / len(dev_tok)
    pruned_dir = os.path.join(OUT_DIR, f"{exp['name']}_pruned")
    model.save_pretrained(pruned_dir); tok.save_pretrained(pruned_dir)
    json.dump({"keep_ids": keep, "original_vocab_size": V, "vocab_top_k": VOCAB_TOP_K, "source_run": os.path.basename(run)},
              open(os.path.join(pruned_dir, "vocab_keep.json"), "w"))
    write_predictions(os.path.join(PRED_DIR, f"{exp['name']}_pruned", "dev_predictions.jsonl"), dev_examples, preds, probs)
    json.dump({"experiment": exp["name"], "pruned_params": pruned_params, "full_params": full_params, "vocab_kept": len(keep),
               "dev_oov_rate": oov, "dev_acc_full": full_acc, "dev_acc_pruned": pruned_acc},
              open(os.path.join(SUM_DIR, f"shrink_{exp['name']}.json"), "w"), indent=2)
    print(f"{exp['name']:<16} vocab {len(keep):,}/{V:,} (dev tokens -> [UNK]: {100 * oov:.2f}%) | "
          f"params {full_params:,} -> {pruned_params:,} | dev acc {full_acc:.4f} -> {pruned_acc:.4f}")
    del model; torch.cuda.empty_cache()

Loading weights:   0%|          | 0/198 [00:00<?, ?it/s]

[transformers] DebertaV2ForSequenceClassification LOAD REPORT from: microsoft/deberta-v3-base
Key                                     | Status     | 
----------------------------------------+------------+-
lm_predictions.lm_head.LayerNorm.weight | UNEXPECTED | 
mask_predictions.LayerNorm.weight       | UNEXPECTED | 
lm_predictions.lm_head.bias             | UNEXPECTED | 
mask_predictions.classifier.weight      | UNEXPECTED | 
mask_predictions.LayerNorm.bias         | UNEXPECTED | 
lm_predictions.lm_head.dense.weight     | UNEXPECTED | 
lm_predictions.lm_head.LayerNorm.bias   | UNEXPECTED | 
lm_predictions.lm_head.dense.bias       | UNEXPECTED | 
mask_predictions.classifier.bias        | UNEXPECTED | 
mask_predictions.dense.bias             | UNEXPECTED | 
mask_predictions.dense.weight           | UNEXPECTED | 
classifier.weight                       | MISSING    | 
pooler.dense.bias                       | MISSING    | 
classifier.bias                         | MISSING    | 
pooler.den

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

base             vocab 20,979/128,100 (dev tokens -> [UNK]: 5.55%) | params 184,423,682 -> 102,154,754 | dev acc 0.8628 -> 0.8087


Loading weights:   0%|          | 0/198 [00:00<?, ?it/s]

[transformers] DebertaV2ForSequenceClassification LOAD REPORT from: microsoft/deberta-v3-xsmall
Key                                        | Status     | 
-------------------------------------------+------------+-
lm_predictions.lm_head.LayerNorm.weight    | UNEXPECTED | 
mask_predictions.LayerNorm.weight          | UNEXPECTED | 
lm_predictions.lm_head.bias                | UNEXPECTED | 
mask_predictions.classifier.weight         | UNEXPECTED | 
mask_predictions.LayerNorm.bias            | UNEXPECTED | 
deberta.embeddings.word_embeddings._weight | UNEXPECTED | 
lm_predictions.lm_head.dense.weight        | UNEXPECTED | 
lm_predictions.lm_head.LayerNorm.bias      | UNEXPECTED | 
lm_predictions.lm_head.dense.bias          | UNEXPECTED | 
mask_predictions.classifier.bias           | UNEXPECTED | 
mask_predictions.dense.bias                | UNEXPECTED | 
mask_predictions.dense.weight              | UNEXPECTED | 
classifier.weight                          | MISSING    | 
pooler.dense.bias  

... (2 outputs left out)

Loading weights:   0%|          | 0/198 [00:00<?, ?it/s]

[transformers] DebertaV2ForSequenceClassification LOAD REPORT from: microsoft/deberta-v3-xsmall
Key                                        | Status     | 
-------------------------------------------+------------+-
lm_predictions.lm_head.LayerNorm.weight    | UNEXPECTED | 
mask_predictions.LayerNorm.weight          | UNEXPECTED | 
lm_predictions.lm_head.bias                | UNEXPECTED | 
mask_predictions.classifier.weight         | UNEXPECTED | 
mask_predictions.LayerNorm.bias            | UNEXPECTED | 
deberta.embeddings.word_embeddings._weight | UNEXPECTED | 
lm_predictions.lm_head.dense.weight        | UNEXPECTED | 
lm_predictions.lm_head.LayerNorm.bias      | UNEXPECTED | 
lm_predictions.lm_head.dense.bias          | UNEXPECTED | 
mask_predictions.classifier.bias           | UNEXPECTED | 
mask_predictions.dense.bias                | UNEXPECTED | 
mask_predictions.dense.weight              | UNEXPECTED | 
classifier.weight                          | MISSING    | 
pooler.dense.bias  

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

xsmall_distill   vocab 20,979/128,100 (dev tokens -> [UNK]: 5.55%) | params 70,830,722 -> 29,696,258 | dev acc 0.7798 -> 0.7545


In [8]:
# ===== 8. COMPARE: accuracy vs size =====
rows = []
for exp in EXPERIMENTS:
    f = os.path.join(SUM_DIR, f"dev_{exp['name']}.json")
    if not os.path.exists(f): continue
    d = json.load(open(f)); s = os.path.join(SUM_DIR, f"shrink_{exp['name']}.json")
    p = json.load(open(s)) if os.path.exists(s) else {}
    rows.append({"experiment": exp["name"], "model": d["model_name"].split("/")[-1], "total_params": d["total_params"],
                 "pruned_params": p.get("pruned_params"), "trainable_params": d["trainable_params"],
                 "n_seeds": len(d["dev_acc_per_run"]), "dev_acc_mean": d["mean"], "dev_acc_std": d["std"],
                 "pruned_dev_acc": p.get("dev_acc_pruned"), "holdout_acc_mean": d["holdout_acc_mean"]})
assert rows, "Nothing to compare yet - run cells 5-7 first."
rows.sort(key=lambda r: r["pruned_params"] or r["total_params"])
print(f"{'experiment':<16}{'params':>13}{'pruned':>13}{'dev acc (mean +- std)':>26}{'pruned dev':>12}{'hold-out':>10}")
for r in rows:
    acc = f"{100 * r['dev_acc_mean']:.2f} +- {100 * r['dev_acc_std']:.2f} (n={r['n_seeds']})"
    pp = f"{r['pruned_params'] / 1e6:.1f}M" if r["pruned_params"] else "-"
    pa = f"{100 * r['pruned_dev_acc']:.2f}" if r["pruned_dev_acc"] is not None else "-"
    print(f"{r['experiment']:<16}{r['total_params'] / 1e6:>12.1f}M{pp:>13}{acc:>26}{pa:>12}{100 * r['holdout_acc_mean']:>10.2f}")
with open("/kaggle/working/accuracy_vs_size.csv", "w", newline="") as f:
    w = csv.DictWriter(f, fieldnames=list(rows[0].keys())); w.writeheader(); w.writerows(rows)
print("\nSaved /kaggle/working/accuracy_vs_size.csv")

experiment             params       pruned     dev acc (mean +- std)  pruned dev  hold-out
xsmall                  70.8M        29.7M       77.26 +- 0.00 (n=1)       72.20     79.52
xsmall_distill          70.8M        29.7M       77.98 +- 0.00 (n=1)       75.45     80.72
base                   184.4M       102.2M       86.28 +- 0.00 (n=1)       80.87     89.56

Saved /kaggle/working/accuracy_vs_size.csv


In [9]:
# ===== 9. TEST predictions with FINAL_EXPERIMENT (pruned model if available) =====
if TEST_FILE:
    test_examples = read_examples(TEST_FILE)
    exp = get_exp(FINAL_EXPERIMENT); pruned_dir = os.path.join(OUT_DIR, f"{exp['name']}_pruned")
    if os.path.isdir(pruned_dir):
        model, tok, meta, remap = load_pruned(pruned_dir); used = os.path.basename(pruned_dir)
    else:
        model, tok, rc = load_trained(chosen_run(exp)); remap = None; used = os.path.basename(chosen_run(exp))
    dl = DataLoader(RTEDataset(test_examples, tok, CONFIG["max_len"]), batch_size=64, collate_fn=make_collate(tok))
    probs = p_entail(predict_logits(model, dl, remap)); preds = [0 if p >= 0.5 else 1 for p in probs]
    write_predictions("/kaggle/working/test_predictions.jsonl", test_examples, preds, probs)
    print(f"{len(preds)} test predictions from {used} -> /kaggle/working/test_predictions.jsonl")
else:
    print("No TEST_FILE - skipping (the course test set is hidden).")

No TEST_FILE - skipping (the course test set is hidden).


In [10]:
# ===== 10. DOWNLOAD: zip everything (download from the Output panel on the right) =====
!cd /kaggle/working && zip -qr rte_outputs.zip runs summary preds accuracy_vs_size.csv *.jsonl 2>/dev/null; ls -lh /kaggle/working

total 551M
-rw-r--r-- 1 root root  475 Nov 10 09:38 accuracy_vs_size.csv
drwxr-xr-x 2 root root 4.0K Nov 10 08:20 data
-rw-r--r-- 1 root root  94K Nov 10 09:38 __notebook__.ipynb
drwxr-xr-x 8 root root 4.0K Nov 10 09:38 preds
-rw-r--r-- 1 root root 551M Nov 10 09:39 rte_outputs.zip
drwxr-xr-x 8 root root 4.0K Nov 10 09:38 runs
drwxr-xr-x 2 root root 4.0K Nov 10 09:38 summary
